In [1]:
import os
os.environ["HF_TOKEN"] = "hf_"

wandb_key = "wandb_v1_"
print(wandb_key)

wandb_v1_


In [ ]:
pip install datasets huggingface_hub transformers sentencepiece tiktoken protobuf pandas pyarrow wandb

In [3]:
from huggingface_hub import hf_hub_download
import pandas as pd

REPO_ID = "akzaidan/Job_Data_Parser"

# Download only the 3 parquet files (no full dataset clone needed)
train_path = hf_hub_download(repo_id=REPO_ID, filename="train.parquet", repo_type="dataset")
val_path   = hf_hub_download(repo_id=REPO_ID, filename="val.parquet",   repo_type="dataset")
test_path  = hf_hub_download(repo_id=REPO_ID, filename="test.parquet",  repo_type="dataset")

print("Downloaded to:")
print("  train:", train_path)
print("  val:  ", val_path)
print("  test: ", test_path)

train.parquet:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

val.parquet:   0%|          | 0.00/61.1M [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/19.9M [00:00<?, ?B/s]

Downloaded to:
  train: /workspace/.cache/huggingface/hub/datasets--akzaidan--Job_Data_Parser/snapshots/1726526eb132e81ad0c74d5f036d882000ea88f7/train.parquet
  val:   /workspace/.cache/huggingface/hub/datasets--akzaidan--Job_Data_Parser/snapshots/1726526eb132e81ad0c74d5f036d882000ea88f7/val.parquet
  test:  /workspace/.cache/huggingface/hub/datasets--akzaidan--Job_Data_Parser/snapshots/1726526eb132e81ad0c74d5f036d882000ea88f7/test.parquet


In [4]:
train_df = pd.read_parquet(train_path)
val_df   = pd.read_parquet(val_path)
test_df  = pd.read_parquet(test_path)

print(f"Train: {len(train_df):,} rows")
print(f"Val:   {len(val_df):,} rows")
print(f"Test:  {len(test_df):,} rows")

Train: 749,313 rows
Val:   29,931 rows
Test:  9,973 rows


In [5]:
print(train_df.columns.tolist())
print(train_df.dtypes)
train_df.head(3)

['text', 'expected_experience_years', 'expected_salary']
text                           str
expected_experience_years    int64
expected_salary              int64
dtype: object


,text,expected_experience_years,expected_salary
0,[LOCATION]: United Kingdom [TITLE]: Facilities...,0,30000
1,[LOCATION]: United States Pennsylvania [TITLE]...,0,35360
2,[LOCATION]: Ireland [TITLE]: Registrar in Haem...,3,95000


In [6]:
import numpy as np
from datasets import Dataset, DatasetDict

# Drop rows where ANY label is -1 (missing)
def drop_missing(df):
    return df[
        (df["expected_experience_years"] != -1) &
        (df["expected_salary"] != -1)
    ].reset_index(drop=True)

train_clean = drop_missing(train_df)
val_clean   = drop_missing(val_df)
test_clean  = drop_missing(test_df)

print(f"Train after dropping missing: {len(train_clean):,}")
print(f"Val after dropping missing:   {len(val_clean):,}")
print(f"Test after dropping missing:  {len(test_clean):,}")

# Log-normalize salary (log1p for safety), z-score normalize experience
# Using ONLY train stats for both

# --- Salary: log then z-score ---
train_clean["expected_salary"] = np.log1p(train_clean["expected_salary"])
val_clean["expected_salary"]   = np.log1p(val_clean["expected_salary"])
test_clean["expected_salary"]  = np.log1p(test_clean["expected_salary"])

TARGET_COLS = ["expected_experience_years", "expected_salary"]
norm_stats = {}

for col in TARGET_COLS:
    mean = train_clean[col].mean()
    std  = train_clean[col].std()
    norm_stats[col] = {"mean": mean, "std": std}
    train_clean[col] = (train_clean[col] - mean) / std
    val_clean[col]   = (val_clean[col]   - mean) / std
    test_clean[col]  = (test_clean[col]  - mean) / std

# Print stats so you can save them for inference later
print("\nnorm_stats (SAVE THESE):")
for col, stats in norm_stats.items():
    print(f"  {col}: mean={stats['mean']:.4f}, std={stats['std']:.4f}")

# Sanity check — all columns should now be centered near 0
print("\nNormalized target distributions:")
print(train_clean[TARGET_COLS].describe().round(3))

# Convert to HuggingFace DatasetDict
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_clean),
    "val":   Dataset.from_pandas(val_clean),
    "test":  Dataset.from_pandas(test_clean),
})

Train after dropping missing: 749,313
Val after dropping missing:   29,931
Test after dropping missing:  9,973

norm_stats (SAVE THESE):
  expected_experience_years: mean=2.9546, std=2.8020
  expected_salary: mean=11.0465, std=0.9250

Normalized target distributions:
       expected_experience_years  expected_salary
count                 749313.000       749313.000
mean                      -0.000           -0.000
std                        1.000            1.000
min                       -1.054          -11.942
25%                       -0.698           -0.444
50%                       -0.341            0.039
75%                        0.730            0.477
max                        6.083            2.244


In [7]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 512
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")
print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

Using device: cuda
GPUs available: 4
  GPU 0: NVIDIA GeForce RTX 5090
  GPU 1: NVIDIA GeForce RTX 5090
  GPU 2: NVIDIA GeForce RTX 5090
  GPU 3: NVIDIA GeForce RTX 5090


In [8]:
import torch

print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version torch was built with: {torch.version.cuda}")

if torch.cuda.is_available():
    x = torch.tensor([1.0]).cuda()
    print(f"Tensor on GPU: {x}")
else:
    print("CUDA not available — check torch build")

Torch version: 2.8.0+cu128
CUDA available: True
CUDA version torch was built with: 12.8
Tensor on GPU: tensor([1.], device='cuda:0')


In [9]:
from transformers import DebertaV2Tokenizer

tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

# Apply tokenizer to all splits
dataset_tokenized = dataset.map(
    tokenize,
    batched=True,
    batch_size=1000,
    desc="Tokenizing"
)

# Set format for PyTorch — include the 2 label columns
dataset_tokenized.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "token_type_ids",
             "expected_experience_years", "expected_salary"]
)

print("Tokenization complete:")
print(dataset_tokenized)

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Tokenizing:   0%|          | 0/749313 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/29931 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/9973 [00:00<?, ? examples/s]

Tokenization complete:
DatasetDict({
    train: Dataset({
        features: ['text', 'expected_experience_years', 'expected_salary', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 749313
    })
    val: Dataset({
        features: ['text', 'expected_experience_years', 'expected_salary', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 29931
    })
    test: Dataset({
        features: ['text', 'expected_experience_years', 'expected_salary', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9973
    })
})


In [10]:
class JobRegressorModel(nn.Module):
    def __init__(self, model_name, dropout=0.2):
        super().__init__()
        # Shared DeBERTa encoder backbone
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size  # 768 for deberta-v3-base
        self.dropout = nn.Dropout(dropout)

        # 2 independent regression heads
        self.head_experience = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )
        self.head_salary = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        # Cast to float32 before regression heads
        cls_embedding = self.dropout(outputs.last_hidden_state[:, 0, :]).float()

        exp    = self.head_experience(cls_embedding).squeeze(-1)
        salary = self.head_salary(cls_embedding).squeeze(-1)

        return exp, salary

# Use all 4 GPUs via DataParallel
model = JobRegressorModel(MODEL_NAME).to(DEVICE).float()
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
    model = nn.DataParallel(model)

print(model)

# Sanity check — count trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using 4 GPUs with DataParallel
DataParallel(
  (module): JobRegressorModel(
    (encoder): DebertaV2Model(
      (embeddings): DebertaV2Embeddings(
        (word_embeddings): Embedding(128100, 768, padding_idx=0)
        (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): DebertaV2Encoder(
        (layer): ModuleList(
          (0-11): 12 x DebertaV2Layer(
            (attention): DebertaV2Attention(
              (self): DisentangledSelfAttention(
                (query_proj): Linear(in_features=768, out_features=768, bias=True)
                (key_proj): Linear(in_features=768, out_features=768, bias=True)
                (value_proj): Linear(in_features=768, out_features=768, bias=True)
                (pos_dropout): Dropout(p=0.1, inplace=False)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): DebertaV2SelfOutput(
                (dense): Linear(

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

In [19]:
import wandb

wandb.login(key=wandb_key)
wandb.init(
    project="job-parser",
    name="deberta-v3-base-run2",
    config={
        "model": "microsoft/deberta-v3-base",
        "epochs": 10,
        "batch_size": 32,
        "learning_rate": 5e-6,
        "max_length": 512,
        "train_samples": len(dataset["train"]),
        "val_samples": len(dataset["val"]),
    }
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [20]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    return {
        "input_ids":                  torch.stack([x["input_ids"] for x in batch]),
        "attention_mask":             torch.stack([x["attention_mask"] for x in batch]),
        "token_type_ids":             torch.stack([x["token_type_ids"] for x in batch]),
        "expected_experience_years":  torch.tensor([x["expected_experience_years"] for x in batch], dtype=torch.float32),
        "expected_salary":            torch.tensor([x["expected_salary"] for x in batch], dtype=torch.float32),
    }

train_loader = DataLoader(dataset_tokenized["train"], batch_size=32, shuffle=True,  collate_fn=collate_fn, num_workers=16, pin_memory=True)
val_loader   = DataLoader(dataset_tokenized["val"],   batch_size=64, shuffle=False, collate_fn=collate_fn, num_workers=16, pin_memory=True)

print(f"Train batches: {len(train_loader):,}")
print(f"Val batches:   {len(val_loader):,}")

Train batches: 23,417
Val batches:   468


In [21]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

EPOCHS       = 10
LR           = 5e-6
WARMUP_STEPS = 2000
TOTAL_STEPS  = len(train_loader) * EPOCHS

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.05)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_STEPS
)
loss_fn = torch.nn.MSELoss()

print(f"Total training steps: {TOTAL_STEPS:,}")
print(f"Warmup steps: {WARMUP_STEPS}")

Total training steps: 234,170
Warmup steps: 2000


In [22]:
import os
from tqdm import tqdm

CHECKPOINT_DIR = "/workspace/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

best_val_loss = float("inf")

for epoch in range(EPOCHS):
    # ── TRAIN ──────────────────────────────────────────────
    model.train()
    train_loss = 0
    train_loss_exp = train_loss_salary = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for step, batch in enumerate(pbar):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        token_type_ids = batch["token_type_ids"].to(DEVICE)
        label_exp      = batch["expected_experience_years"].to(DEVICE)
        label_salary   = batch["expected_salary"].to(DEVICE)

        optimizer.zero_grad()

        pred_exp, pred_salary = model(input_ids, attention_mask, token_type_ids)
        loss_exp    = loss_fn(pred_exp,    label_exp)
        loss_salary = loss_fn(pred_salary, label_salary)
        loss        = loss_exp + loss_salary

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        train_loss        += loss.item()
        train_loss_exp    += loss_exp.item()
        train_loss_salary += loss_salary.item()

        # Log to wandb every 100 steps
        if step % 100 == 0:
            wandb.log({
                "train/loss":        loss.item(),
                "train/loss_exp":    loss_exp.item(),
                "train/loss_salary": loss_salary.item(),
                "train/lr":          scheduler.get_last_lr()[0],
                "step":              epoch * len(train_loader) + step,
            })

        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_train_loss = train_loss / len(train_loader)

    # ── VALIDATE ───────────────────────────────────────────
    model.eval()
    val_loss = 0
    val_loss_exp = val_loss_salary = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            token_type_ids = batch["token_type_ids"].to(DEVICE)
            label_exp      = batch["expected_experience_years"].to(DEVICE)
            label_salary   = batch["expected_salary"].to(DEVICE)

            pred_exp, pred_salary = model(input_ids, attention_mask, token_type_ids)
            loss_exp    = loss_fn(pred_exp,    label_exp)
            loss_salary = loss_fn(pred_salary, label_salary)
            loss        = loss_exp + loss_salary

            val_loss        += loss.item()
            val_loss_exp    += loss_exp.item()
            val_loss_salary += loss_salary.item()

    avg_val_loss        = val_loss        / len(val_loader)
    avg_val_loss_exp    = val_loss_exp    / len(val_loader)
    avg_val_loss_salary = val_loss_salary / len(val_loader)

    # Log epoch-level metrics to wandb
    wandb.log({
        "epoch":                  epoch + 1,
        "train/epoch_loss":       avg_train_loss,
        "val/epoch_loss":         avg_val_loss,
        "val/loss_exp":           avg_val_loss_exp,
        "val/loss_salary":        avg_val_loss_salary,
    })

    print(f"\nEpoch {epoch+1} Summary:")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val Loss:   {avg_val_loss:.4f}  (exp={avg_val_loss_exp:.4f}, salary={avg_val_loss_salary:.4f})")

    # ── SAVE BEST CHECKPOINT ───────────────────────────────
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        checkpoint_path = f"{CHECKPOINT_DIR}/best_model.pt"
        torch.save({
            "epoch":                epoch + 1,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss":             best_val_loss,
            "norm_stats":           norm_stats,
        }, checkpoint_path)
        print(f"  ✓ New best model saved (val_loss={best_val_loss:.4f})")
        wandb.save(checkpoint_path)

wandb.finish()
print("\nTraining complete!")

Epoch 1/10 [Val]: 100%|██████████| 468/468 [00:55<00:00,  8.43it/s]



Epoch 1 Summary:
  Train Loss: 0.5660
  Val Loss:   0.6846  (exp=0.1081, salary=0.5766)


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


  ✓ New best model saved (val_loss=0.6846)


Epoch 2/10 [Val]: 100%|██████████| 468/468 [00:55<00:00,  8.46it/s]


...


Epoch 5 Summary:
  Train Loss: 0.4519
  Val Loss:   0.6825  (exp=0.1038, salary=0.5787)


Epoch 6/10 [Val]: 100%|██████████| 468/468 [00:56<00:00,  8.36it/s]



Epoch 6 Summary:
  Train Loss: 0.4336
  Val Loss:   0.6959  (exp=0.1050, salary=0.5909)


Epoch 7/10 [Train]:   2%|▏         | 430/23417 [01:21<1:12:45,  5.27it/s, loss=0.2713]


KeyboardInterrupt: 

In [ ]:
import numpy as np
from tqdm import tqdm

test_loader = DataLoader(dataset_tokenized["test"], batch_size=64, shuffle=False, collate_fn=collate_fn, num_workers=16, pin_memory=True)

# Load best checkpoint
checkpoint = torch.load(f"{CHECKPOINT_DIR}/best_model.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
norm_stats = checkpoint["norm_stats"]
print(f"Loaded best model from epoch {checkpoint['epoch']} (val_loss={checkpoint['val_loss']:.4f})")

# ── RUN TEST SET ───────────────────────────────────────────
model.eval()

all_pred_exp,    all_label_exp    = [], []
all_pred_salary, all_label_salary = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        token_type_ids = batch["token_type_ids"].to(DEVICE)

        pred_exp, pred_salary = model(input_ids, attention_mask, token_type_ids)

        all_pred_exp.append(pred_exp.cpu().numpy())
        all_pred_salary.append(pred_salary.cpu().numpy())

        all_label_exp.append(batch["expected_experience_years"].numpy())
        all_label_salary.append(batch["expected_salary"].numpy())

# Flatten
pred_exp    = np.concatenate(all_pred_exp)
pred_salary = np.concatenate(all_pred_salary)

label_exp    = np.concatenate(all_label_exp)
label_salary = np.concatenate(all_label_salary)

# ── DENORMALIZE ────────────────────────────────────────────
def denorm(arr, col):
    return arr * norm_stats[col]["std"] + norm_stats[col]["mean"]

def denorm_salary(arr):
    return np.expm1(denorm(arr, "expected_salary"))

pred_exp_real    = denorm(pred_exp,    "expected_experience_years")
pred_salary_real = denorm_salary(pred_salary)

label_exp_real    = denorm(label_exp,    "expected_experience_years")
label_salary_real = denorm_salary(label_salary)

# Pull texts directly from the dataset
all_texts = [dataset_tokenized["test"][int(i)]["text"] for i in range(len(pred_exp_real))]

# ── METRICS ────────────────────────────────────────────────
def mae(pred, label):         return np.mean(np.abs(pred - label))
def rmse(pred, label):        return np.sqrt(np.mean((pred - label) ** 2))
def within_n(pred, label, n): return np.mean(np.abs(pred - label) <= n) * 100

results = {
    "exp_mae_years":      mae(pred_exp_real,    label_exp_real),
    "exp_rmse_years":     rmse(pred_exp_real,   label_exp_real),
    "exp_within_1yr":     within_n(pred_exp_real,    label_exp_real,    1),
    "exp_within_2yr":     within_n(pred_exp_real,    label_exp_real,    2),
    "salary_mae_usd":     mae(pred_salary_real,  label_salary_real),
    "salary_rmse_usd":    rmse(pred_salary_real, label_salary_real),
    "salary_within_10k":  within_n(pred_salary_real, label_salary_real, 10_000),
    "salary_within_20k":  within_n(pred_salary_real, label_salary_real, 20_000),
}

# ── PRINT ──────────────────────────────────────────────────
print("\n══════════════════════════════════════════")
print("           TEST SET RESULTS")
print("══════════════════════════════════════════")
print(f"\n  Experience Years:")
print(f"    MAE:            {results['exp_mae_years']:.2f} years")
print(f"    RMSE:           {results['exp_rmse_years']:.2f} years")
print(f"    Within 1 year:  {results['exp_within_1yr']:.1f}%")
print(f"    Within 2 years: {results['exp_within_2yr']:.1f}%")
print(f"\n  Expected Salary:")
print(f"    MAE:            ${results['salary_mae_usd']:,.0f}")
print(f"    RMSE:           ${results['salary_rmse_usd']:,.0f}")
print(f"    Within $10k:    {results['salary_within_10k']:.1f}%")
print(f"    Within $20k:    {results['salary_within_20k']:.1f}%")
print("══════════════════════════════════════════")

# ── LOG TO WANDB ───────────────────────────────────────────
wandb.init(project="job-parser", name="deberta-v3-base-run1", resume="allow")
wandb.log({"test/" + k: v for k, v in results.items()})
wandb.finish()

# ── FIRST 5 PREDICTIONS ────────────────────────────────────
print("\n  Sample Predictions vs Ground Truth (first 5):")
print(f"  {'#':>3} {'Exp(pred)':>10} {'Exp(true)':>10} {'Salary(pred)':>14} {'Salary(true)':>14}")
for i in range(5):
    print(f"  {i+1:>3} {pred_exp_real[i]:>10.1f} {label_exp_real[i]:>10.1f} "
          f"${pred_salary_real[i]:>13,.0f} ${label_salary_real[i]:>13,.0f}")

# ── 20 RANDOM PREDICTIONS WITH TEXT PREVIEW ───────────────
print("\n══════════════════════════════════════════════════════════════════════════════════════════════════════════")
print("           20 RANDOM PREDICTIONS")
print("══════════════════════════════════════════════════════════════════════════════════════════════════════════")
print(f"  {'#':>3}  {'Text Preview':<80}  {'Exp(pred)':>9} {'Exp(true)':>9} {'Salary(pred)':>13} {'Salary(true)':>13}")
print(f"  {'':-<3}  {'':-<80}  {'':-<9} {'':-<9} {'':-<13} {'':-<13}")

random_indices = np.random.choice(len(pred_exp_real), size=20, replace=False)
for i, idx in enumerate(random_indices):
    preview = all_texts[idx][:80].replace("\n", " ")
    print(f"  {i+1:>3}  {preview:<80}  "
          f"{pred_exp_real[idx]:>9.1f} {label_exp_real[idx]:>9.1f} "
          f"${pred_salary_real[idx]:>12,.0f} ${label_salary_real[idx]:>12,.0f}")

print("══════════════════════════════════════════════════════════════════════════════════════════════════════════")

Loaded best model from epoch 3 (val_loss=0.6588)


Testing:  11%|█         | 17/156 [00:05<00:16,  8.61it/s]

In [24]:
# ── INFERENCE ──────────────────────────────────────────────
JOB_TITLE = "Early Software Engineer"
JOB_DESCRIPTION = """
We are looking for a lead Software Engineer to join our team.
You will be responsible for designing and implementing scalable backend services.
Requirements: Strong experience with Python, distributed systems, and cloud infrastructure.
"""

# ── RUN THROUGH MODEL ──────────────────────────────────────
model.eval()
text = f"[LOCATION] Remote [TITLE]: {JOB_TITLE} [DESC]: {JOB_DESCRIPTION.strip()}"
encoded = tokenizer(
    text,
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt"
)
input_ids      = encoded["input_ids"].to(DEVICE)
attention_mask = encoded["attention_mask"].to(DEVICE)
token_type_ids = encoded["token_type_ids"].to(DEVICE)

with torch.no_grad():
    pred_exp, pred_salary = model(input_ids, attention_mask, token_type_ids)

# ── DENORMALIZE ────────────────────────────────────────────
exp_years      = pred_exp.item()    * norm_stats["expected_experience_years"]["std"] + norm_stats["expected_experience_years"]["mean"]
salary         = np.expm1(pred_salary.item() * norm_stats["expected_salary"]["std"] + norm_stats["expected_salary"]["mean"])

# ── PRINT RESULT ───────────────────────────────────────────
print("══════════════════════════════════════════")
print("           JOB CLASSIFICATION")
print("══════════════════════════════════════════")
print(f"  Title:               {JOB_TITLE}")
print(f"  Expected Experience: {max(0, round(exp_years))} years")
print(f"  Expected Salary:     ${max(0, salary):,.0f}")
print("══════════════════════════════════════════")

══════════════════════════════════════════
           JOB CLASSIFICATION
══════════════════════════════════════════
  Title:               Early Software Engineer
  Expected Experience: 3 years
  Expected Salary:     $85,461
══════════════════════════════════════════


In [25]:
import json
from huggingface_hub import HfApi, create_repo

HF_USERNAME = "akzaidan"
REPO_NAME   = "JobPredictor2"
REPO_ID     = f"{HF_USERNAME}/{REPO_NAME}"

# ── CREATE REPO ────────────────────────────────────────────
api = create_repo(REPO_ID, repo_type="model", exist_ok=True)
print(f"Repo ready: https://huggingface.co/{REPO_ID}")

# ── SAVE MODEL FILES LOCALLY ───────────────────────────────
UPLOAD_DIR = "/workspace/JobPredictor1"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# 1. Save model weights
torch.save(model.state_dict(), f"{UPLOAD_DIR}/pytorch_model.bin")

# 2. Save norm_stats (needed for inference)
with open(f"{UPLOAD_DIR}/norm_stats.json", "w") as f:
    json.dump(norm_stats, f, indent=2)

# 3. Save tokenizer
tokenizer.save_pretrained(UPLOAD_DIR)

# 4. Save model config so anyone knows what base model was used
config_data = {
    "base_model":   "microsoft/deberta-v3-base",
    "architecture": "DeBERTa-v3-base + 2 regression heads",
    "outputs": {
        "expected_experience_years": "integer (years of experience)",
        "expected_salary":           "integer (expected salary USD)",
    },
    "normalization": {
        "expected_experience_years": "z-score — use norm_stats.json to denormalize",
        "expected_salary":           "log1p then z-score — reverse with: expm1(pred * std + mean)"
    },
    "max_length":    512,
    "dropout":       0.2,
}
with open(f"{UPLOAD_DIR}/config.json", "w") as f:
    json.dump(config_data, f, indent=2)

# 5. Save a model card (README)
model_card = """---
language: en
tags:
  - job-classification
  - salary-prediction
  - experience-prediction
  - deberta
---
# JobPredictor1
Fine-tuned DeBERTa-v3-base model that predicts:
- **Expected years of experience** required for a job
- **Expected salary** (USD)

## Input Format
```
[LOCATION] <Remote | United States (State) | Country> [TITLE]: <job title> [DESC]: <job description>
```

## Outputs
| Output | Type | Description |
|---|---|---|
| expected_experience_years | int | Years of experience required |
| expected_salary | int | Expected salary (USD) |

## Normalization
Experience is z-score normalized:
```python
real_value = pred * norm_stats["expected_experience_years"]["std"] + norm_stats["expected_experience_years"]["mean"]
```
Salary is log1p + z-score normalized:
```python
real_salary = np.expm1(pred * norm_stats["expected_salary"]["std"] + norm_stats["expected_salary"]["mean"])
```

## Test Set Performance
| Metric | Value |
|---|---|
| Experience MAE | 0.57 years |
| Experience Within 1yr | 83.1% |
| Salary MAE | $15,511 |
| Salary Within $20k | 84.5% |

## Base Model
microsoft/deberta-v3-base
"""
with open(f"{UPLOAD_DIR}/README.md", "w") as f:
    f.write(model_card)

print("All files saved locally:")
for fname in os.listdir(UPLOAD_DIR):
    print(f"  {fname}")

# ── UPLOAD TO HUGGINGFACE ──────────────────────────────────
api = HfApi()
api.upload_folder(
    folder_path=UPLOAD_DIR,
    repo_id=REPO_ID,
    repo_type="model"
)
print(f"\n✓ Model uploaded to: https://huggingface.co/{REPO_ID}")

Repo ready: https://huggingface.co/akzaidan/JobPredictor2
All files saved locally:
  pytorch_model.bin
  norm_stats.json
  tokenizer_config.json
  tokenizer.json
  config.json
  README.md


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


✓ Model uploaded to: https://huggingface.co/akzaidan/JobPredictor2
